# Assignment 2 — ReAct Agent with a Search Tool

This notebook completes the Agent Loop Demo assignment. It uses Gemini function calling to build a **ReAct agent** that can reason, call a document-search tool, read the tool result, and then answer.

## What the agent does
1. Receives a question.
2. Decides whether to call `search_documents`.
3. Searches the document collection and returns relevant passages.
4. Reads the observation and produces a final answer.
5. Says when the supplied documents do not contain the answer.

## Before running
- Add your Gemini API key in Colab **Secrets** using the name `GOOGLE_API_KEY`.
- Turn **Notebook access** ON for that secret.
- Use **Runtime → Run all**.

The API key is read from Colab Secrets and is not written into this notebook. Do not paste the key into a code cell or upload it to GitHub.


## 1. Install the Google Gen AI SDK

This notebook uses only the current `google-genai` SDK. It does not mix the older `google-generativeai` package with the newer SDK.

In [1]:
%pip -q install -U google-genai
print("Google Gen AI SDK is ready.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.1 which is incompatible.
google-adk 2.7.1 requires google-genai<3,>=2.12.1, but you have google-genai 3.0.0 which is incompatible.
Google Gen AI SDK is ready.


## 2. Load the API key securely

In Colab, open the **Secrets** panel (key icon), add a secret named `GOOGLE_API_KEY`, paste your key as its value, and enable notebook access. If the notebook is run outside Colab, it will ask for the key privately.

In [2]:
import os
import getpass as _getpass_module

api_key = None

# First try the Colab Secrets panel.
try:
    from google.colab import userdata
    secret_value = userdata.get("GOOGLE_API_KEY")

    # Normally Colab returns a string. Handle dictionary-shaped values safely too.
    if isinstance(secret_value, dict):
        for possible_key in ("GOOGLE_API_KEY", "value", "secret", "api_key"):
            candidate = secret_value.get(possible_key)
            if isinstance(candidate, str) and candidate.strip():
                api_key = candidate.strip()
                break
    elif isinstance(secret_value, str):
        api_key = secret_value.strip()

except Exception as error:
    print("Could not read the Colab Secret automatically:", error)

# If the secret is missing or inaccessible, ask privately.
if not api_key:
    entered_value = _getpass_module.getpass(
        "Enter your Gemini API key (input is hidden): "
    )

    if isinstance(entered_value, dict):
        for possible_key in ("GOOGLE_API_KEY", "value", "secret", "api_key"):
            candidate = entered_value.get(possible_key)
            if isinstance(candidate, str) and candidate.strip():
                api_key = candidate.strip()
                break
    elif isinstance(entered_value, str):
        api_key = entered_value.strip()

if not isinstance(api_key, str) or not api_key:
    raise ValueError(
        "No valid API key was received. In Colab, open the Secrets panel, "
        "add GOOGLE_API_KEY, enable Notebook access, and rerun this cell."
    )

print("API key loaded successfully. The key itself will not be displayed.")


API key loaded successfully. The key itself will not be displayed.


## 3. Create the document collection

The assignment allows documents on any topic. These four documents cover cybersecurity, Linux, networking, and Python. Each document contains at least four sentences.

In [3]:
import re
import time

DOCUMENTS = {
    "cybersecurity_basics": (
        "Cybersecurity is the practice of protecting computers, networks, applications, and data "
        "from digital attacks. A strong password should be long, unique, and difficult to guess. "
        "Multi-factor authentication adds another layer of protection by requiring a second form "
        "of verification. Software updates are important because they often fix security weaknesses. "
        "Phishing is a social engineering attack in which someone tries to trick a person into "
        "revealing sensitive information or opening a malicious link."
    ),
    "linux_fundamentals": (
        "Linux is an open-source operating system widely used on servers, desktops, and embedded devices. "
        "The terminal allows users to interact with the system by entering commands. The pwd command "
        "prints the current working directory, while ls lists files and directories. The cd command "
        "changes the current directory. File permissions determine who can read, write, or execute a file, "
        "and the chmod command is commonly used to modify those permissions."
    ),
    "computer_networking": (
        "A computer network connects devices so that they can exchange data and share resources. "
        "An IP address identifies a device on an Internet Protocol network. DNS translates human-readable "
        "domain names into IP addresses that computers use to communicate. HTTP is a protocol used for "
        "transferring web resources, while HTTPS protects HTTP communication using encryption through TLS. "
        "A router forwards packets between different networks."
    ),
    "python_programming": (
        "Python is a high-level programming language known for its readable syntax and large ecosystem "
        "of libraries. A list is an ordered and mutable collection that can store multiple values. "
        "A dictionary stores information as key-value pairs and is useful for looking up values by key. "
        "Functions are defined with the def keyword and help organise reusable code. Exceptions can be "
        "handled with try and except blocks so that a program can respond to errors without stopping unexpectedly."
    )
}

# Assignment validation: at least four documents, each with at least four sentences.
assert len(DOCUMENTS) >= 4
for document_name, document_text in DOCUMENTS.items():
    sentence_count = len(re.findall(r"[.!?]+", document_text))
    assert sentence_count >= 4, f"{document_name} needs at least four sentences."
    print(f"{document_name}: {sentence_count} sentences")

print(f"\nDocument collection ready: {len(DOCUMENTS)} documents.")

cybersecurity_basics: 5 sentences
linux_fundamentals: 5 sentences
computer_networking: 5 sentences
python_programming: 5 sentences

Document collection ready: 4 documents.


## 4. Define the search tool

`search_documents` ranks documents by the number of meaningful query words they contain and returns the two best matches. This is the action the Gemini agent can choose to call.

In [4]:
STOP_WORDS = {
    "the", "is", "a", "an", "and", "or", "to", "of", "in", "on", "for",
    "with", "what", "how", "why", "does", "do", "are", "can", "i", "it",
    "that", "this", "from", "by", "be", "used", "use", "which", "when",
    "into", "their", "your", "account"
}

def search_documents(query: str) -> str:
    """Search the document collection and return up to two relevant passages."""
    query_terms = {
        word for word in re.findall(r"[a-zA-Z0-9]+", query.lower())
        if word not in STOP_WORDS and len(word) > 1
    }

    if not query_terms:
        return "Please provide a more specific search query."

    ranked_results = []
    for document_name, content in DOCUMENTS.items():
        document_terms = set(re.findall(r"[a-zA-Z0-9]+", content.lower()))
        matched_terms = query_terms.intersection(document_terms)
        score = len(matched_terms)

        if score:
            ranked_results.append((score, document_name, content, matched_terms))

    ranked_results.sort(key=lambda item: item[0], reverse=True)

    if not ranked_results:
        return "No relevant documents found for that query."

    output = []
    for score, name, content, matched_terms in ranked_results[:2]:
        output.append(
            f"[Source: {name} | Match score: {score} | "
            f"Matched terms: {', '.join(sorted(matched_terms))}]\n{content}"
        )

    return "\n\n".join(output)


# Quick sanity check of the search tool.
print(search_documents("DNS translates domain names into IP addresses"))

[Source: computer_networking | Match score: 6 | Matched terms: addresses, dns, domain, ip, names, translates]
A computer network connects devices so that they can exchange data and share resources. An IP address identifies a device on an Internet Protocol network. DNS translates human-readable domain names into IP addresses that computers use to communicate. HTTP is a protocol used for transferring web resources, while HTTPS protects HTTP communication using encryption through TLS. A router forwards packets between different networks.


## 5. Configure Gemini and declare the tool

The model can call `search_documents` by supplying a query. Automatic tool execution is disabled because the notebook implements the ReAct loop explicitly, so the reasoning/action/observation steps are visible.

In [5]:
from google import genai
from google.genai import types

client = genai.Client(api_key=api_key)

# Stable Gemini model first, with stable Flash fallbacks for temporary capacity errors.
MODEL_CANDIDATES = [
    "gemini-3.8-flash",
    "gemini-3.5-flash",
    "gemini-3.5-flash-lite",
]

search_function = types.FunctionDeclaration(
    name="search_documents",
    description=(
        "Search the assignment's document collection for passages relevant to a question. "
        "Use this when the answer may be present in the supplied documents."
    ),
    parameters=types.Schema(
        type=types.Type.OBJECT,
        properties={
            "query": types.Schema(
                type=types.Type.STRING,
                description="A concise search query describing the information needed."
            )
        },
        required=["query"]
    )
)

search_tool = types.Tool(function_declarations=[search_function])

SYSTEM_INSTRUCTION = """
You are a helpful question-answering agent with access to a search_documents tool.
The tool searches a small collection of reference documents supplied with this notebook.

Follow these rules:
1. Understand the user's question.
2. If the answer may be in the supplied documents, call search_documents.
3. Read the returned search result before answering.
4. Answer clearly and concisely, based on the retrieved documents.
5. If the documents do not contain the answer, say that the information is not available
   in the supplied documents. Do not invent facts or pretend you searched the web.
6. You may make another search call if the first result is insufficient.
"""

generation_config = types.GenerateContentConfig(
    system_instruction=SYSTEM_INSTRUCTION,
    tools=[search_tool],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
    temperature=0.2
)

print("Gemini client and document-search tool are configured.")
print("Primary model:", MODEL_CANDIDATES[0])

Gemini client and document-search tool are configured.
Primary model: gemini-3.8-flash


## 6. Add retry and model fallback handling

Google may temporarily return `429` (rate limit) or `503` (server busy). The function below retries temporary errors with short waits and then tries the next stable Flash model. Authentication or code errors are not hidden; they are shown so they can be fixed.

In [6]:
def generate_with_retry(contents, preferred_model=None, retries_per_model=2):
    """Generate a response, retrying transient errors and trying fallback models."""
    if preferred_model and preferred_model in MODEL_CANDIDATES:
        model_order = [preferred_model] + [
            name for name in MODEL_CANDIDATES if name != preferred_model
        ]
    else:
        model_order = MODEL_CANDIDATES[:]

    last_error = None

    for model_name in model_order:
        for attempt in range(retries_per_model):
            try:
                response = client.models.generate_content(
                    model=model_name,
                    contents=contents,
                    config=generation_config
                )
                return response, model_name

            except Exception as error:
                last_error = error
                message = str(error).lower()

                transient_error = any(token in message for token in [
                    "429", "resource_exhausted",
                    "500", "502", "503", "504",
                    "internal", "unavailable", "deadline", "temporarily"
                ])

                if not transient_error:
                    raise

                if attempt < retries_per_model - 1:
                    wait_seconds = 5 * (attempt + 1)
                    print(
                        f"{model_name} is temporarily unavailable. "
                        f"Retrying in {wait_seconds}s..."
                    )
                    time.sleep(wait_seconds)
                else:
                    print(f"{model_name} is busy; trying the next model if available.")

    raise RuntimeError(
        "Gemini models are temporarily unavailable or rate-limited. "
        "Wait a few minutes and rerun the cell. Last error: " + str(last_error)
    )


print("Retry and fallback handling is ready.")

Retry and fallback handling is ready.


## 7. Implement the ReAct agent loop

The loop alternates between:
- **Reason:** Gemini decides what information it needs.
- **Act:** Gemini requests the search tool and Python executes it.
- **Observe:** The tool result is sent back to Gemini.
- **Final answer:** Gemini responds using the observation.

The model's response content is appended unchanged to preserve any function-calling metadata.

In [7]:
def run_agent(question: str, max_steps: int = 6) -> str:
    """Run a manual ReAct loop for one user question."""
    print(f"Question: {question}")
    print("=" * 72)

    contents = [
        types.Content(
            role="user",
            parts=[types.Part.from_text(text=question)]
        )
    ]

    active_model = None

    for step in range(1, max_steps + 1):
        response, active_model = generate_with_retry(
            contents,
            preferred_model=active_model
        )

        if not response.candidates:
            raise RuntimeError("Gemini returned no response candidates.")

        model_content = response.candidates[0].content

        # Preserve the model turn exactly as returned.
        contents.append(model_content)

        function_calls = [
            part.function_call
            for part in model_content.parts
            if getattr(part, "function_call", None) is not None
        ]

        # No function call means the model has produced its final answer.
        if not function_calls:
            final_answer = response.text or "The model did not return a text answer."
            print(f"\nFinal answer ({active_model}):\n{final_answer}")
            return final_answer

        # Execute all requested calls and return the observations together.
        function_response_parts = []

        for function_call in function_calls:
            if function_call.name != "search_documents":
                tool_result = f"Unknown tool: {function_call.name}"
            else:
                arguments = dict(function_call.args or {})
                query = arguments.get("query", "")
                print(f"\nStep {step} — ACT: search_documents({query!r})")
                tool_result = search_documents(query)
                print("OBSERVATION:")
                print(tool_result[:900])

            # Do not pass an id argument here: the installed SDK's
            # Part.from_function_response signature does not accept it.
            function_response_parts.append(
                types.Part.from_function_response(
                    name=function_call.name,
                    response={"result": tool_result}
                )
            )

        contents.append(
            types.Content(role="user", parts=function_response_parts)
        )

    print("\nAgent stopped after reaching the maximum number of steps.")
    return "The agent reached its step limit before producing a final answer."


print("ReAct agent is ready.")

ReAct agent is ready.


## 8. Run one initial test

This test asks about multi-factor authentication. The agent should call the search tool, receive the cybersecurity document, and answer using that information.

In [8]:
_ = run_agent("How does multi-factor authentication help protect an account?")

Question: How does multi-factor authentication help protect an account?
gemini-3.8-flash is temporarily unavailable. Retrying in 5s...
gemini-3.8-flash is busy; trying the next model if available.

Step 1 — ACT: search_documents('multi-factor authentication help protect account')
OBSERVATION:
[Source: cybersecurity_basics | Match score: 3 | Matched terms: authentication, factor, multi]
Cybersecurity is the practice of protecting computers, networks, applications, and data from digital attacks. A strong password should be long, unique, and difficult to guess. Multi-factor authentication adds another layer of protection by requiring a second form of verification. Software updates are important because they often fix security weaknesses. Phishing is a social engineering attack in which someone tries to trick a person into revealing sensitive information or opening a malicious link.

[Source: python_programming | Match score: 1 | Matched terms: help]
Python is a high-level programming lang

## 9. Ask at least four follow-up questions

These tests cover:
- Questions answered by the supplied documents.
- Questions about different documents.
- A question that cannot be answered from the documents.
- A general-knowledge question that is also outside the document collection.

The agent is instructed not to invent answers when the documents do not contain the information.

In [9]:
questions = [
    "What is the purpose of DNS and what does it translate?",
    "Which Linux commands show the current directory and list files?",
    "How many ghosts eat carrots every day?",
    "What is the most popular food item in Canada?"
]

assert len(questions) >= 4
assert all(question.strip() for question in questions)

for index, question in enumerate(questions, start=1):
    print("\n" + "#" * 72)
    print(f"TEST QUESTION {index}/{len(questions)}")
    print("#" * 72)

    run_agent(question)

    # Keep requests spaced out to reduce free-tier rate limiting.
    if index < len(questions):
        time.sleep(7)


########################################################################
TEST QUESTION 1/4
########################################################################
Question: What is the purpose of DNS and what does it translate?
gemini-3.8-flash is temporarily unavailable. Retrying in 5s...
gemini-3.8-flash is busy; trying the next model if available.

Step 1 — ACT: search_documents('DNS domain name system translate purpose')
OBSERVATION:
[Source: computer_networking | Match score: 2 | Matched terms: dns, domain]
A computer network connects devices so that they can exchange data and share resources. An IP address identifies a device on an Internet Protocol network. DNS translates human-readable domain names into IP addresses that computers use to communicate. HTTP is a protocol used for transferring web resources, while HTTPS protects HTTP communication using encryption through TLS. A router forwards packets between different networks.

[Source: linux_fundamentals | Match score: 1 | M

## 10. Final assignment validation

Run this cell after the tests. It checks that the notebook has at least four documents, each document has at least four sentences, and there are at least four non-empty test questions.

In [10]:
assert len(DOCUMENTS) >= 4, "Add at least four documents."

for document_name, document_text in DOCUMENTS.items():
    sentence_count = len(re.findall(r"[.!?]+", document_text))
    assert sentence_count >= 4, f"{document_name} has fewer than four sentences."

assert len(questions) >= 4, "Add at least four test questions."
assert all(question.strip() for question in questions), "A test question is empty."

print("All assignment validation checks passed.")
print(f"Documents: {len(DOCUMENTS)}")
print(f"Test questions: {len(questions)}")
print("API key is not stored in the notebook.")


All assignment validation checks passed.
Documents: 4
Test questions: 4
API key is not stored in the notebook.
